# Create GTEX_TCGA friendly version of elife data

Andrew E. Davidson  
aedaivds@ucsc.edu 02/10/25  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0  

create Data sets that make it easy to work with HUGO GTEx_TCGA and  v39 Ensembl gene id elife


This data can be use to evaluate a random forest multiclassifier or train a GAN

In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
import math
import numpy as np
import os
import pandas as pd
import pprint as pp
import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

#outDir = f'{notebookDir}/{notebookName}.out'
outDir = f'/private/groups/kimlab/aedavids/elife/{notebookName}.out'
os.makedirs(outDir, exist_ok=True)
print(f'outDir:\n{outDir}')

dataOutDir = os.path.join(outDir, "data")
os.makedirs(dataOutDir, exist_ok=True)
print(f'\ndataOutDir ;\n{dataOutDir}')

import logging
#loglevel = "DEBUG"
#loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

meaningOfLife = 42

outDir:
/private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out

dataOutDir ;
/private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data


In [2]:
# setting the python path allows us to run python scripts from using
# the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src


sys.p

In [3]:
notebookPath.parent

PosixPath('/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife')

In [4]:
# local imports
from analysis.utilities import saveList
from intraExtraRNA.elifeUtilities import loadElifeTrainingData
from models.mlUtilities import saveLabelEncoder

In [5]:
%%time

pipelineStageName = "best10CuratedDegree1_ce467ff"

selectElifeCategories = [ "Colorectal Cancer", "Esophagus Cancer", "Healthy donor",
                         "Liver Cancer", "Lung Cancer", "Stomach Cancer"]

# Whole_Blood is considered a healthy control
# the binary classifiers extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/elifeBinaryRandomForestResults.ipynb 
# use a single feature. for example "COAD" and did not include features from a healthy control
# the thought was 'COAD or not. Keep the number of features to a min'.
# the 'Lung" featues was a lump of see /private/groups/kimlab/GTEx_TCGA/1vsAllLumpBrain and 
# extraCellularRNA/deconvolutionAnalysis/lumpBrain
features = ["COAD", "ESCA",  "Esophagus_Mucosa", "Liver", "Lung", "Stomach", "Whole_Blood"]

t = loadElifeTrainingData(pipelineStageName,
                                 features,
                                 selectElifeCategories,
                                 )
HUGO_Genes, elifeGenes, missingGenes, countDF, metaDF, XDF, yNP, labelEncoder, mapDF = t

2025-02-18 16:25:50,929 WARNING intraExtraRNA.elifeUtilities selectFeatures() line:420] [missingInV39Set: {'LTR106'}]


CPU times: user 1min 6s, sys: 10.3 s, total: 1min 16s
Wall time: 1min 18s


In [6]:
print(f'len(HUGO_Genes) : {len(HUGO_Genes)}')
print(f'len(elifeGenes) : {len(elifeGenes)}')
print( f'missingGenes : {missingGenes}' )

print(f'\ncountDF.shape : {countDF.shape}')
print(f'countDF.iloc[0:5, 0:5]')
display(countDF.iloc[0:5, 0:5]) 

print(f'\n metaDF.shape : {metaDF.shape}')
print(f'metaDF.iloc[0:5, 0:5]')
display(metaDF.iloc[0:5, :]) 

print(f'\n XDF.shape : {XDF.shape}')
print(f'XDF.iloc[0:5, 0:5]')
display(XDF.iloc[0:5, :]) 


print(f'\nyNP.shape : {yNP.shape}')

print(f'\nmapDF.shape : {mapDF.shape}')
mapDF.iloc[0:5, :]

len(HUGO_Genes) : 70
len(elifeGenes) : 70
missingGenes : []

countDF.shape : (224, 76555)
countDF.iloc[0:5, 0:5]


gene,(A)n,(AAA)n,(AAAAAAC)n,(AAAAAAG)n,(AAAAAAT)n
SRR14506659,201.672053,0.0,0.0,0.0,0.0
SRR14506660,110.450773,0.0,0.0,0.0,0.0
SRR14506661,3722.776395,0.0,0.0,0.0,0.0
SRR14506662,1394.605651,0.0,0.0,0.0,0.0
SRR14506663,2843.473730,0.0,0.0,0.0,0.0



 metaDF.shape : (224, 2)
metaDF.iloc[0:5, 0:5]


,sample_id,diagnosis
0,SRR14506659,Esophagus Cancer
1,SRR14506660,Esophagus Cancer
2,SRR14506661,Esophagus Cancer
3,SRR14506662,Esophagus Cancer
4,SRR14506663,Esophagus Cancer



 XDF.shape : (224, 70)
XDF.iloc[0:5, 0:5]


gene,ENSG00000117395.13,ENSG00000158714.11,ENSG00000180667.11,ENSG00000170385.10,ENSG00000134318.15,ENSG00000225889.10,ENSG00000183607.10,ENSG00000138443.17,ENSG00000144596.13,ENSG00000182247.10,...,ENSG00000187474.5,ENSG00000167555.14,ENSG00000063241.8,ENSG00000131845.15,ENSG00000198934.5,ENSG00000203952.9,(TA)n,MER5C,HERVFH19-int,LTR106_Mam
SRR14506659,74.463527,0.000000,1697.147893,65.155586,564.681749,0.000000,0.0,161.337643,43.437058,40.334411,...,0.0,15.513235,15.513235,0.000000,0.000000,0.0,0.000000,80.668821,124.105879,9.307941
SRR14506660,229.237454,36.816924,566.147046,368.169245,40.984878,0.000000,0.0,527.940804,42.374196,77.107144,...,0.0,64.603283,0.000000,289.672783,18.061133,0.0,0.000000,0.000000,0.000000,0.000000
SRR14506661,0.000000,0.000000,940.677707,0.000000,199.672155,48.808749,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,199.672155,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
SRR14506662,520.926228,0.000000,356.854975,0.000000,319.938943,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
SRR14506663,577.672082,0.000000,1018.732524,81.966985,58.547846,25.370733,0.0,236.142980,117.095692,66.354226,...,0.0,17.564354,0.000000,0.000000,0.000000,0.0,7.806379,95.628149,70.257415,0.000000



yNP.shape : (224,)

mapDF.shape : (70, 3)


,HUGO_v35,ENSG_v35,ENSG_v39
0,EBNA1BP2,ENSG00000117395.13,ENSG00000117395.13
1,SLAMF8,ENSG00000158714.11,ENSG00000158714.11
2,YOD1,ENSG00000180667.10,ENSG00000180667.11
3,SLC30A1,ENSG00000170385.10,ENSG00000170385.10
4,ROCK2,ENSG00000134318.14,ENSG00000134318.15


# Save Data sets

In [7]:
dfDict = {
    "countDF" : countDF , 
    "metaDF"  : metaDF , 
    "XDF"     : XDF,  
    "mapDF"  : mapDF 
    }

for key,df in dfDict.items():
    p = f'{dataOutDir}/{key}.csv'
    countDF.to_csv(p, index=False)
    print(f'saved : {p}')

saved : /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/countDF.csv
saved : /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/metaDF.csv
saved : /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/XDF.csv
saved : /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/mapDF.csv


In [8]:
listDict = {
    'HUGO_Genes'   : HUGO_Genes, 
    'elifeGenes'   : elifeGenes, 
    'missingGenes' : missingGenes
}

for key,l in listDict.items():
    p = f'{dataOutDir}/{key}.txt'
    saveList(p, l)
    print(f'saved {p}')

saved /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/HUGO_Genes.txt
saved /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/elifeGenes.txt
saved /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/missingGenes.txt


In [9]:
labelEncoderPath = f'{dataOutDir}/labelEncoder.dict'
saveLabelEncoder(labelEncoderPath, labelEncoder)
print(f'\nlabelEncoder saved to :\n {labelEncoderPath}')


labelEncoder saved to :
 /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/labelEncoder.dict
